In [ ]:
# Установка всех внешних библиотек для пайплайна (Python 3.13.9, 64-bit).
# Запустить ОДИН раз в новом окружении Jupyter, затем перезапустить kernel.
# Важно: устанавливаем пакеты в интерпретатор, используемый этим ноутбуком.
import sys
import platform
import subprocess
from importlib.metadata import version, PackageNotFoundError

if sys.version_info[:2] != (3, 13):
    raise RuntimeError(
        f"Требуется Python 3.13.x (проверено для 3.13.9); "
        f"текущая версия: {sys.version.split()[0]}"
    )
if sys.maxsize <= 2**32:
    raise RuntimeError("Необходим 64-битный Python для PyTorch.")

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)

# Выбери способ установки PyTorch:
# 'cpu'   — работает без NVIDIA GPU (самый универсальный вариант)
# 'cu126' — NVIDIA GPU + подходящий драйвер CUDA 12.6
# 'cu130' — NVIDIA GPU + подходящий драйвер CUDA 13.0
# 'cu132' — NVIDIA GPU + подходящий драйвер CUDA 13.2
TORCH_BACKEND = 'cpu'


def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '--prefer-binary', *packages]
    print('\n>', ' '.join(cmd), flush=True)
    subprocess.check_call(cmd)


# Обновление инструментов установки.
pip_install('--upgrade', 'pip', 'setuptools', 'wheel')

# Численные расчёты, таблицы, изображения, графики, PCA, Jupyter.
pip_install(
    'numpy>=2.1,<3',
    'pandas>=2.2,<4',
    'matplotlib>=3.9,<4',
    'seaborn>=0.13.2,<1',
    'scikit-learn>=1.6,<2',
    'Pillow>=11,<13',
    'opencv-contrib-python>=4.11,<5',  # cv2.aruco + OpenCV GUI
    'jupyterlab>=4,<5',
    'ipykernel>=6.29,<8',
)

# Совместимая пара PyTorch/TorchVision с поддержкой Python 3.13.
torch_packages = ('torch==2.14.1', 'torchvision==0.29.1')
if platform.system() in ('Windows', 'Linux'):
    if TORCH_BACKEND not in ('cpu', 'cu126', 'cu130', 'cu132'):
        raise ValueError('TORCH_BACKEND: cpu, cu126, cu130 или cu132')
    pip_install(*torch_packages,
                '--index-url', f'https://download.pytorch.org/whl/{TORCH_BACKEND}')
elif TORCH_BACKEND == 'cpu':
    # macOS: обычные пакеты PyPI (в том числе Apple Silicon).
    pip_install(*torch_packages)
else:
    raise ValueError('CUDA-варианты этого блока предназначены для Windows/Linux с NVIDIA GPU')

# Совместимая с текущим кодом SegFormer ветка Transformers 4.x.
pip_install('transformers==4.57.6', 'safetensors>=0.5,<1')

print('\nУстановленные версии:')
for package in ('numpy', 'pandas', 'opencv-contrib-python', 'matplotlib',
                'seaborn', 'scikit-learn', 'Pillow', 'torch',
                'torchvision', 'transformers', 'safetensors', 'ipykernel'):
    try:
        print(f'  {package}: {version(package)}')
    except PackageNotFoundError:
        print(f'  {package}: НЕ УСТАНОВЛЕН')

print('\nГотово. Перезапусти Jupyter kernel (Kernel → Restart Kernel),')
print('после чего выполняй остальные ячейки сверху вниз.')
print('Модель GlobalWheat/GWFSS_Segformer_b0 будет скачана при первом запуске сегментации.')

In [ ]:
from pathlib import Path
import traceback
import logging
from datetime import datetime
import pandas as pd
from process_wheat_study import process_wheat_study
from calculate_vegetation_indices import calculate_vegetation_indices, INDEX_INFO
from plot_spectral_pca import plot_spectral_pca
from create_vegetation_index_maps import create_vegetation_index_maps
from create_study_collages import create_study_collages

# Все служебные метки можно изменить для нового эксперимента.
MARKERS = dict(
    calibration_marker="calibration_target",  # ArUco-мишень
    spectralon_marker="spectralon",           # Малый Spectralon, отдельная съемка (необязательно при большом листе)
    flat_marker="flat",                       # Серый фон ИЛИ большой Spectralon-лист
    dark_marker="dark",                       # Темновые кадры; та же экспозиция/усиление
    exclude_marker="#",                       # Исключить ветку дерева каталогов
)

# Коррекции включены по умолчанию. Включать только при наличии нужных папок.
# reuse_existing=True удобно для ДОПРОЦЕССИНГА после сбоя: уже готовые файлы
# будут переиспользованы, а недостающие этапы пересчитаны.
# overwrite_existing=True принудительно пересоздаёт все производные файлы.
OPTIONS = dict(
    use_dark_correction=True,     # ПЕРВАЯ операция: RAW - DARK
    use_alignment=True,           # Аффинное совмещение по ArUco
    use_spatial_correction=True,  # Пространственная коррекция по flat
    use_spectral_correction=True, # Нормировка по Spectralon (rho=1)
    flat_is_spectralon=False,     # True: весь flat — большой Spectralon; отдельная папка spectralon не нужна
    reference_wavelength=700,
    spectralon_wavelength=400,
    spectralon_edge_margin_px=5,
    min_flat_fraction=0.01,
    save_float_cube=True,
    cube_dtype="float16",        # Сохраняем в float16; считаем в float32
    preview_percentiles=(1, 99),  # Только preview-изображения для просмотра и сегментации
    rgb_wavelengths=(630, 530, 470),
    stretch_percentiles=(1, 99),
    classes=("leaf",),
    class_thresholds={"leaf": 0.60, "stem": 0.60, "head": 0.60},
    tile_size=2048,
    overlap=512,
    reuse_existing=False,         # True: использовать уже готовые результаты, если они есть
    overwrite_existing=False,     # True: пересоздать всё заново; имеет приоритет над reuse_existing
    continue_on_error=True,
    show_plot=True,
)

# Все доступные индексы автоматически.
INDEX_NAMES = list(INDEX_INFO.keys())

# Единая цветовая шкала по каждому индексу; файл лежит рядом с модулями.
# В начале используем черновые границы. После завершения всех образцов
# последняя ячейка пересоздаёт карты и коллажи по общим эмпирическим границам.
COLOR_SCALE_CSV = None            # None: index_color_ranges.csv рядом с Python-модулями
EMPIRICAL_STRATEGY = "robust"      # "robust" (min P2 / max P98) или "extrema" (min/max)
# v5: быстрые карты через Pillow и небольшой кэш полос (без копий всего куба).
INDEX_MAP_RENDERER = "fast"        # "fast" или "matplotlib" (прежний стиль, медленнее)
INDEX_BAND_CACHE = 12              # число спектральных полос в кэше на один образец



def _study_logger(main):
    logger = logging.getLogger("wheat_batch." + str(main))
    logger.setLevel(logging.INFO)
    logger.propagate = False
    if not logger.handlers:
        handler = logging.FileHandler(main / "study_stages.log", encoding="utf-8", mode="a")
        handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        logger.addHandler(handler)
    return logger


def process_main_folder(main_folder):
    """Обработка исследования. Ошибка этапа не останавливает следующий этап/папку."""
    main = Path(main_folder).expanduser().resolve()
    logger = _study_logger(main)
    print(f"\n==== {main.name} ====", flush=True)
    reuse = OPTIONS.get("reuse_existing", False) and not OPTIONS.get("overwrite_existing", False)
    overwrite = not reuse
    kind = "reflectance" if OPTIONS["use_spectral_correction"] else "signal"
    stage_rows = []
    result = {}

    def attempt(stage, func, can_skip=False, skip=None):
        try:
            if reuse and can_skip:
                obj = skip() if skip is not None else None
                state = "reused"
            else:
                obj = func()
                state = "ok"
            stage_rows.append(dict(stage=stage, status=state, error=""))
            logger.info("%s: %s", stage, state)
            return obj
        except Exception as exc:
            detail = traceback.format_exc()
            logger.error("%s FAILED: %s\n%s", stage, exc, detail)
            stage_rows.append(dict(stage=stage, status="error", error=f"{type(exc).__name__}: {exc}"))
            print(f"ERROR {main.name} / {stage}: {exc}")
            return None
        finally:
            pd.DataFrame(stage_rows).to_csv(main / "study_stages_report.csv", index=False)

    processed = attempt("wheat_pipeline", lambda: process_wheat_study(
        main_folder=main, **MARKERS, **OPTIONS))
    if processed is not None:
        df_mean, df_std, df_long, report = processed
        result.update(df_mean=df_mean, df_std=df_std, df_long=df_long, report=report)
    else:
        # Calibration or other shared stages may fail; still allow reusing old spectra.
        report_path = main / "processing_report.csv"
        if report_path.is_file():
            result["report"] = pd.read_csv(report_path)

    spectra_csv = main / "spectra_mean.csv"
    # Downstream processing requires a usable per-study mean spectrum table.
    has_spectra = spectra_csv.is_file() and spectra_csv.stat().st_size > 0
    if not has_spectra:
        logger.warning("No spectra_mean.csv: skipping indices and PCA")
        for stage in ("indices", "pca"):
            stage_rows.append(dict(stage=stage, status="skipped", error="spectra_mean.csv unavailable"))
    else:
        indices_csv = main / "vegetation_indices.csv"
        metadata_csv = main / "vegetation_indices_descriptions.csv"
        # Сводные таблицы/статистика всегда обновляются по свежей spectra_mean.csv.
        # Этот этап быстрый, зато при возвращении исправленного образца
        # автоматически появится новая строка в таблице индексов.
        result["indices"] = attempt(
            "indices",
            lambda: calculate_vegetation_indices(
                spectra_csv, output_csv=indices_csv,
                metadata_csv=metadata_csv, data_type=kind),
        )
        pca_files = [main / n for n in (
            "wheat_pca.png", "wheat_pca_scores.csv", "cluster_diagnostics.csv")]
        result["pca"] = attempt(
            "pca",
            lambda: plot_spectral_pca(
                spectra_csv=spectra_csv, output_png=pca_files[0],
                output_scores_csv=pca_files[1],
                output_cluster_scores_csv=pca_files[2],
                n_clusters="auto", cluster_range=(2, 8),
                scaling="center", normalize_spectra=True,
                annotate=True, show=False),
        )

    result["maps_report"] = attempt("index_maps", lambda: create_vegetation_index_maps(
        main_folder=main, corrected_folder_name="corrected_images",
        indices=INDEX_NAMES, data_type=kind, use_mask=True,
        overwrite=overwrite, color_config_csv=COLOR_SCALE_CSV,
        scale_source="draft", empirical_strategy=EMPIRICAL_STRATEGY,
        renderer=INDEX_MAP_RENDERER, max_cached_bands=INDEX_BAND_CACHE, **MARKERS))
    # Коллажи — лёгкий агрегирующий этап. Перестраиваем их даже при reuse=True,
    # поскольку список образцов и количество готовых карт могли измениться.
    result["collage_report"] = attempt("collages", lambda: create_study_collages(
        main_folder=main, index_maps_folder="index_maps", indices=INDEX_NAMES,
        n_columns=4, tile_width=360, tile_height=310, show=False,
        overwrite=overwrite,
        **{k: v for k, v in MARKERS.items() if k != "dark_marker"}))

    pd.DataFrame(stage_rows).to_csv(main / "study_stages_report.csv", index=False)
    result["stage_report"] = pd.DataFrame(stage_rows)
    return result


In [ ]:
# Все непосредственные подпапки текущей директории
working_dir = Path.cwd()
folder_list = sorted((p for p in working_dir.iterdir()
    if p.is_dir() and not p.name.startswith("#")), key=lambda x:x.name.casefold())
for i, folder in enumerate(folder_list):
    print(i, folder.name)


In [ ]:
# Отредактируй срез или явный выбор индексов.
selected_folders = folder_list[:]
# selected_folders = folder_list[:-1]
# selected_folders = folder_list[1:4]
# selected_folders = [folder_list[i] for i in (0, 2, 4)]
print("Будут обработаны:", [p.name for p in selected_folders])



In [ ]:
# Пакетный запуск с журналом ошибок по главным папкам.
results, failed_folders, batch_rows = {}, {}, []
working_dir = Path.cwd()
for index, folder in enumerate(selected_folders, 1):
    print(f"[{index}/{len(selected_folders)}] {folder.name}", flush=True)
    try:
        result = process_main_folder(folder)
        results[folder.name] = result
        stages = result.get("stage_report")
        stage_errors = 0 if stages is None else int((stages["status"] == "error").sum())
        samples = result.get("report")
        sample_errors = 0 if samples is None or "status" not in samples else int((samples["status"] == "error").sum())
        status = "ok" if not (stage_errors or sample_errors) else "partial"
        batch_rows.append(dict(folder=str(folder), status=status,
                               sample_errors=sample_errors, stage_errors=stage_errors, error=""))
    except Exception as exc:
        failed_folders[folder.name] = f"{type(exc).__name__}: {exc}"
        batch_rows.append(dict(folder=str(folder), status="error",
                               sample_errors=0, stage_errors=0,
                               error=failed_folders[folder.name]))
        with (working_dir / "wheat_batch_errors.log").open("a", encoding="utf-8") as log:
            log.write(f"\n[{datetime.now().isoformat()}] {folder}\n{traceback.format_exc()}\n")
        print("ERROR", failed_folders[folder.name])
    finally:
        pd.DataFrame(batch_rows).to_csv(working_dir / "wheat_batch_report.csv", index=False)

print("Полностью успешно:", sum(r["status"] == "ok" for r in batch_rows))
print("Частично успешно:", sum(r["status"] == "partial" for r in batch_rows))
print("Ошибки папок:", len(failed_folders))


In [ ]:
# FINAL STEP (optional): redraw ALL index maps and collages using the
# study-wide empirical bounds after every possible sample has been processed.
# This does not redo dark correction, geometric alignment, SegFormer, or cubes.
# Uses existing corrected_cube.npz and cached empirical statistics when current.
# If cubes/masks changed, statistics are refreshed before maps are rendered.
# If an index has policy="fixed" (for example NDVI 0–1), it stays fixed.

def rebuild_final_maps_and_collages(main_folder):
    main = Path(main_folder).expanduser().resolve()
    kind = "reflectance" if OPTIONS["use_spectral_correction"] else "signal"
    print(f"Finalizing common index scales: {main.name}")
    maps_report = create_vegetation_index_maps(
        main_folder=main,
        corrected_folder_name="corrected_images",
        indices=INDEX_NAMES,
        data_type=kind,
        use_mask=True,
        color_config_csv=COLOR_SCALE_CSV,
        scale_source="empirical",
        empirical_strategy=EMPIRICAL_STRATEGY,
        overwrite=True,
        renderer=INDEX_MAP_RENDERER,
        max_cached_bands=INDEX_BAND_CACHE,
        **MARKERS,
    )
    # IMPORTANT: collages must be recreated AFTER updating the maps.
    collages_report = create_study_collages(
        main_folder=main,
        index_maps_folder="index_maps",
        indices=INDEX_NAMES,
        n_columns=4, tile_width=360, tile_height=310,
        overwrite=True, show=False,
        **{k:v for k,v in MARKERS.items() if k != "dark_marker"},
    )
    print("Completed:", main.name)
    return maps_report, collages_report

# Use the selected_folders list from cell 4.
finalization_status = {}
for folder in selected_folders:
    try:
        maps_report, collages_report = rebuild_final_maps_and_collages(folder)
        n_errors = int((maps_report["status"] == "error").sum())
        if n_errors:
            raise RuntimeError(f"{n_errors} failed map entries; see vegetation_index_maps_report.csv")
        finalization_status[folder.name] = "ok"
    except Exception as exc:
        finalization_status[folder.name] = f"{type(exc).__name__}: {exc}"
        print("ERROR:", folder.name, finalization_status[folder.name])

display(pd.DataFrame([
    {"study": name, "status": status}
    for name,status in finalization_status.items()
]))
